# ROGII V5 - Oil & Gas Geology AI Prediction

## Based on V4: Particle Filter + Spatial Formation Prior + Calibrated Blend + Projection Smooth + Visualization

- V4 Core: FormationPrior, GR Particle Filter, Multi-model, Calibrate Blend
- V5 New: Full pipeline visualization, Optimized params, Detailed intermediate results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os, re, time, json, pickle
from pathlib import Path
from glob import glob
from collections import defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Sequence

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('darkgrid')

from scipy.spatial import cKDTree
print('Environment Ready')

In [ ]:
# ==================== V5 Global Config ====================
class Config:
    DATA_DIR = Path(r'D:\PyCharm\kaggle_competition_train\rogii-wellbore-geology-prediction')
    OUTPUT_DIR = Path(r'D:\PyCharm\kaggle_competition_train\5\output')
    MODEL_DIR = Path(r'D:\PyCharm\kaggle_competition_train\5\models')
    
    TARGET = 'TVT'
    SUB_TARGET = 'tvt'
    
    FORMATIONS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']
    COMMON_NUMERIC = ['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input']
    
    PF_TRAIN_PARTICLES = 128
    PF_TRAIN_SEEDS = 3
    PF_TEST_PARTICLES = 256
    PF_TEST_SEEDS = 5
    
    MAX_TRAIN_ROWS = 1_200_000
    CALIBRATION_ROWS = 300_000
    VALIDATION_FRACTION = 0.18
    MAX_TRAIN_WELLS = None
    
    PROJECTION_BLEND = 0.65
    PROJECTION_DEGREE = 4
    
    SEED = 42
    TAIL_WINDOWS = [25, 50, 100, 300, 1000]

cfg = Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data Dir: {cfg.DATA_DIR}')
print(f'Output Dir: {cfg.OUTPUT_DIR}')
print(f'Model Dir: {cfg.MODEL_DIR}')
print(f'PF Train Particles: {cfg.PF_TRAIN_PARTICLES}, PF Test Particles: {cfg.PF_TEST_PARTICLES}')